# Bike Sharing in the Bay Area

We end this chapter by using all the methods we have learned to examine a new and large dataset. We will also draw maps of the bike stations.

The [Bay Area Bike Share](http://www.bayareabikeshare.com/) service published a [dataset](http://www.bayareabikeshare.com/open-data) describing every bicycle rental from September 2014 to August 2015 in their system. There were 354,152 rentals in all. The columns are:

- An ID for the rental
- Duration of the rental, in seconds
- Start date
- Name of the Start Station and code for Start Terminal
- Name of the End Station and code for End Terminal
- A serial number for the bike
- Subscriber type and zip code

In [ ]:
import polars as pl
import hvplot.polars
from IPython.display import display
path_data = '../../../assets/data/'
import math
from scipy import stats
import folium


In [ ]:
# POLARS_CHANGE: some zip codes are the text "nil", so that column is read as strings
trips = pl.read_csv(path_data + "trip.csv", schema_overrides={"Zip Code": pl.String})
trips


We'll focus only on the *free trips*, which are trips that last less than 1800 seconds (half an hour). There is a charge for longer trips.

The histogram below shows that most of the trips took around 10 minutes (600 seconds) or so. Very few took near 30 minutes (1800 seconds), possibly because people try to return the bikes before the cutoff time so as not to have to pay.

In [ ]:
commute = trips.filter(pl.col("Duration") < 1800)
commute.hvplot.hist(
    "Duration",
    bins=10,
    normed=True,
    xlabel="Duration (Second)",
)


We can get more detail by specifying a larger number of bins. But the overall shape doesn't change much.

In [ ]:
commute.hvplot.hist(
    "Duration",
    bins=60,
    normed=True,
    xlabel="Duration (Second)",
)


## Exploring the Data with `group` and `pivot`

We can use `group` to identify the most highly used Start Station:

In [ ]:
starts = commute.group_by("Start Station").len(name="count").sort("count", descending=True)
starts


The largest number of trips started at the Caltrain Station on Townsend and 4th in San Francisco. People take the train into the city, and then use a shared bike to get to their next destination.

The `group` method can also be used to classify the rentals by both Start Station and End Station.

In [ ]:
commute.group_by(["Start Station", "End Station"]).len(name="count").sort("count", descending=True)


Fifty-four trips both started and ended at the station on 2nd at Folsom. A much large number (437) were between 2nd at Folsom and 2nd at Townsend. 

The `pivot` method does the same classification but displays its results in a contingency table that shows all possible combinations of Start and End Stations, even though some of them didn't correspond to any trips. Remember that the first argument of `pivot` specifies the column labels of the pivot table; `index` labels the rows.

There is a train station as well as a Bay Area Rapid Transit (BART) station near Beale at Market, explaining the high number of trips that start and end there.

In [ ]:
station_counts = commute.pivot("Start Station", index="End Station", aggregate_function="len")
value_cols = sorted(col for col in station_counts.columns if col != "End Station")
station_counts.select("End Station", *value_cols).sort("End Station")


We can also use `pivot` to find the shortest time of the rides between Start and End Stations. Here `pivot` has been given `Duration` as the optional `values` argument, and `min` as the function which to perform on the values in each cell.

In [ ]:
shortest = commute.pivot(
    "Start Station",
    index="End Station",
    values="Duration",
    aggregate_function="min",
)
value_cols = sorted(col for col in shortest.columns if col != "End Station")
shortest.select("End Station", *value_cols).sort("End Station")


Someone had a very quick trip (271 seconds, or about 4.5 minutes) from Beale at Market to 2nd at Folsom, about five blocks away. There are no bike trips between the 2nd Avenue stations and Adobe on Almaden, because the latter is in a different city.

## Drawing Maps
The DataFrame `stations` contains geographical information about each bike station, including latitude, longitude, and a "landmark" which is the name of the city where the station is located.

In [ ]:
stations = pl.read_csv(path_data + "station.csv")
stations


We can draw a map of where the stations are located. The DataFrame's columns are, in order, latitude, longitude, and an optional identifier for each point.

In [ ]:
# POLARS_CHANGE: Polars does not draw maps, so folium places one marker per station
points = stations.select("lat", "long", "name").rename({"name": "labels"})
station_map = folium.Map(
    location=[points.get_column("lat").mean(), points.get_column("long").mean()],
    zoom_start=10,
)
for lat, lon, name in points.iter_rows():
    folium.Marker([lat, lon], popup=name).add_to(station_map)
station_map


The map is created using [OpenStreetMap](http://www.openstreetmap.org/#map=5/51.500/-0.100), which is an open online mapping system that you can use just as you would use Google Maps or any other online map. Zoom in to San Francisco to see how the stations are distributed. Click on a marker to see which station it is.

You can also represent points on a map by colored circles. Here is such a map of the San Francisco bike stations.

In [ ]:
sf = stations.filter(pl.col("landmark") == "San Francisco")
sf_map_data = sf.select("lat", "long", "name").rename({"name": "labels"})
sf_map = folium.Map(
    location=[sf_map_data.get_column("lat").mean(), sf_map_data.get_column("long").mean()],
    zoom_start=13,
)
for lat, lon, name in sf_map_data.iter_rows():
    folium.CircleMarker([lat, lon], popup=name, color="green", fill=True, radius=10).add_to(sf_map)
sf_map


## More Informative Maps: An Application of `join`
The bike stations are located in five different cities in the Bay Area. To distinguish the points by using a different color for each city, let's start by using group to identify all the cities and assign each one a color.

In [ ]:
# POLARS_CHANGE: group_by does not sort, so cities are sorted before the colors are assigned
cities = stations.group_by("landmark").len(name="count").sort("landmark").rename({"landmark": "city"})
cities


In [ ]:
colors = cities.with_columns(
    pl.Series("color", ["blue", "red", "green", "orange", "purple"])
)
colors


Now we can join `stations` and `colors` by `landmark`, and then select the columns we need to draw a map.

In [ ]:
joined = stations.join(colors, left_on="landmark", right_on="city")
colored = joined.select("lat", "long", "name", "color").rename({"name": "labels"})
colored_map = folium.Map(
    location=[colored.get_column("lat").mean(), colored.get_column("long").mean()],
    zoom_start=10,
)
for lat, lon, name, color in colored.iter_rows():
    folium.Marker([lat, lon], popup=name, icon=folium.Icon(color=color)).add_to(colored_map)
colored_map


Now the markers have five different colors for the five different cities.

To see where most of the bike rentals originate, let's identify the start stations:

In [ ]:
starts = commute.group_by("Start Station").len(name="count").sort("count", descending=True)
starts


We can include the geographical data needed to map these stations, by first joining `starts` with `stations`:

In [ ]:
station_starts = stations.join(starts, left_on="name", right_on="Start Station").sort("name")
station_starts


Now we extract just the data needed for drawing our map, adding a color and an area to each station. The area is 0.3 times the count of the number of rentals starting at each station, where the constant 0.3 was chosen so that the circles would appear at an appropriate scale on the map.

In [ ]:
starts_map_data = station_starts.select("lat", "long", "name").with_columns(
    pl.lit("blue").alias("colors"),
    (station_starts.get_column("count") * 0.3).alias("areas"),
).rename({"name": "labels"})
display(starts_map_data.head(3))
starts_map = folium.Map(
    location=[starts_map_data.get_column("lat").mean(), starts_map_data.get_column("long").mean()],
    zoom_start=10,
)
for lat, lon, name, color, area in starts_map_data.iter_rows():
    folium.CircleMarker(
        [lat, lon],
        popup=name,
        color=color,
        fill=True,
        radius=(area / math.pi) ** 0.5,
    ).add_to(starts_map)
starts_map


That huge blob in San Francisco shows that the eastern section of the city is the unrivaled capital of bike rentals in the Bay Area.